In [1]:
import duckdb

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [3]:
df = con.execute("SELECT * FROM bronze_produtos").df()

In [ ]:
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_20261117202546.csv,2026-11-17 20:45:56.094179
1,10002,MARTELO,BT50,100,1500,z0019_20261117202546.csv,2026-11-17 20:45:56.094179
2,10003,PREGO,BT10,100,50,z0019_20261117202546.csv,2026-11-17 20:45:56.094179
3,10004,SERRA,BT50,100,200,z0019_20261117202849.csv,2026-11-17 20:51:26.076199
4,10005,MACHADO,BT50,100,100,z0019_20261117202849.csv,2026-11-17 20:51:26.076199
5,10003,PREGO,BT10,100,60,z0019_20261117202849.csv,2026-11-17 20:51:26.076199


In [ ]:
df.sort_values(by='data_ingestao', ascending=False, inplace=True)

In [7]:
df.drop_duplicates(subset=['NATBR'], keep='first', inplace=True)

In [16]:
df = con.execute("""
        SELECT * FROM (
            SELECT 
                 *,
                ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row_num
            FROM bronze_produtos
                WHERE data_ingestao >= '2026-11-17')
        WHERE row_num = 1
""").df()

In [18]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row_num'])
df_final.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [20]:
rename_dict = {
    'NATBR' : 'id',
    'MAKTX' : 'nm_produto',
    'WERKS' : 'id_categoria',
    'MAINS' : 'id_fornecedor',
    'LABST' : 'vl_preco'
}

df_final.rename(columns=rename_dict, inplace=True)
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [ ]:
df2 = df_final
df2 = df2.astype({
    'id'            : 'int32',
    'nm_produto'    : 'string',
    'id_categoria'  : 'string',
    'id_fornecedor' : 'int32',
    'vl_preco'      : 'float32'
})

df2.dtypes

id                        int32
nm_produto       string[python]
id_categoria     string[python]
id_fornecedor             int32
vl_preco                float32
dtype: object

In [ ]:
con.execute('''
CREATE TABLE IF NOT EXISTS produtos(
    id              INT     ,
    nm_produto      STRING  ,
    id_categoria    STRING  ,
    id_fornecedor   INT     ,
    vl_preco        FLOAT
)
''')

In [27]:
con.execute('''insert into produtos select * from df2''')

In [28]:
df_resultado = con.execute("SELECT * FROM produtos").df()
df_resultado.head()

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [30]:
con.close()